# RePaint Dataset Conversion and Training

This notebook prepares synthetic glass and wood defects for every image in the dataset located at `/home/jovyan/work/Ayan/repaint` and trains a diffusion-based RePaint model on the converted data.

**Workflow overview**

1. Convert the raw dataset by injecting `1` glass defect and `2` wood defects per image, generating per-class masks, a combined inpainting mask, and the context images required by RePaint.
2. Build PyTorch dataloaders over the converted dataset.
3. Fine-tune a UNet denoiser with a RePaint-compatible diffusion scheduler.
4. Run qualitative inference to validate the trained model.

Adjust the configuration cells (paths, image size, hyper-parameters) to match your hardware budget before launching the training section.


In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet pillow numpy pandas matplotlib tqdm opencv-python-headless scikit-image torchvision torch diffusers accelerate datasets


In [ ]:
import math
import os
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm import tqdm

import matplotlib.pyplot as plt
from IPython.display import display

RAW_DATA_ROOT = Path('/home/jovyan/work/Ayan/repaint').expanduser()
if not RAW_DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset path {RAW_DATA_ROOT} does not exist. Update RAW_DATA_ROOT before proceeding.')

CONVERTED_ROOT = RAW_DATA_ROOT / 'converted_repaint'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'Raw dataset root : {RAW_DATA_ROOT.resolve()}')
print(f'Converted dataset: {CONVERTED_ROOT.resolve()}')


## Dataset Conversion

The helper functions below synthesise glass and wood defects, generate the associated masks, and build the RePaint-ready dataset structure.


In [ ]:
def random_bbox(width, height, min_ratio=0.12, max_ratio=0.28, rng=None):
    rng = rng or random
    defect_w = max(8, int(width * rng.uniform(min_ratio, max_ratio)))
    defect_h = max(8, int(height * rng.uniform(min_ratio, max_ratio)))
    x0 = rng.randint(0, max(0, width - defect_w))
    y0 = rng.randint(0, max(0, height - defect_h))
    return x0, y0, x0 + defect_w, y0 + defect_h

def generate_glass_mask(width, height, rng):
    mask = np.zeros((height, width), dtype=np.float32)
    x0, y0, x1, y1 = random_bbox(width, height, rng=rng)
    center = (int((x0 + x1) / 2), int((y0 + y1) / 2))
    axes = (max(6, int((x1 - x0) / 2)), max(6, int((y1 - y0) / 2)))
    angle = rng.uniform(0, 180)
    cv2.ellipse(mask, center, axes, angle, 0, 360, 1, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=rng.uniform(2.5, 4.0), sigmaY=rng.uniform(2.5, 4.0))
    return np.clip(mask, 0.0, 1.0)

def generate_wood_mask(width, height, rng, min_vertices=6, max_vertices=10):
    mask = np.zeros((height, width), dtype=np.float32)
    num_vertices = rng.randint(min_vertices, max_vertices)
    center_x = rng.uniform(0.2, 0.8) * width
    center_y = rng.uniform(0.2, 0.8) * height
    base_radius = rng.uniform(0.12, 0.25) * min(width, height)
    points = []
    for i in range(num_vertices):
        angle = (2 * math.pi * i) / num_vertices + rng.uniform(-0.1, 0.1)
        radius = base_radius * rng.uniform(0.7, 1.35)
        x = int(np.clip(center_x + radius * math.cos(angle), 0, width - 1))
        y = int(np.clip(center_y + radius * math.sin(angle), 0, height - 1))
        points.append([x, y])
    points = np.array(points, dtype=np.int32)
    if points.shape[0] >= 3:
        cv2.fillPoly(mask, [points], 1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=rng.uniform(1.2, 2.2), sigmaY=rng.uniform(1.2, 2.2))
    return np.clip(mask, 0.0, 1.0)

def apply_glass_effect(image_np, mask, rng):
    mask_expanded = mask[..., None]
    strength = rng.uniform(0.45, 0.65)
    tint = np.array([rng.uniform(210, 240), rng.uniform(220, 250), 255.0], dtype=np.float32)
    softened = cv2.GaussianBlur(image_np, (0, 0), sigmaX=rng.uniform(0.8, 1.6))
    highlight = softened * (1 - strength * mask_expanded) + tint * (strength * mask_expanded)
    glare = cv2.GaussianBlur(highlight, (0, 0), sigmaX=rng.uniform(0.5, 1.5))
    return np.clip(glare, 0, 255)

def apply_wood_effect(image_np, mask, rng):
    mask_expanded = mask[..., None]
    texture_noise = rng.normal(0, 18, size=image_np.shape).astype(np.float32)
    warm_tone = np.array([rng.uniform(90, 130), rng.uniform(60, 90), rng.uniform(40, 70)], dtype=np.float32)
    blended = image_np * (1 - mask_expanded) + (image_np * 0.55 + warm_tone + texture_noise) * mask_expanded
    return np.clip(blended, 0, 255)

def inject_defects(image_path, rng, glass_count=1, wood_count=2, min_pixels=96):
    image = Image.open(image_path).convert('RGB')
    width, height = image.size
    base_np = np.array(image, dtype=np.float32)
    defect_np = base_np.copy()
    combined_mask = np.zeros((height, width), dtype=np.uint8)
    glass_mask = np.zeros((height, width), dtype=np.uint8)
    wood_mask = np.zeros((height, width), dtype=np.uint8)

    for g_idx in range(glass_count):
        for _ in range(12):
            mask = generate_glass_mask(width, height, rng)
            binary = (mask > 0.35).astype(np.uint8)
            if binary.sum() < min_pixels:
                continue
            overlap_fraction = (
                (combined_mask > 0).astype(np.uint8)[binary == 1].sum() / (binary.sum() + 1e-6)
            )
            if overlap_fraction > 0.2:
                continue
            effect_mask = mask * binary
            defect_np = apply_glass_effect(defect_np, effect_mask, rng)
            combined_mask = np.where(binary == 1, 1, combined_mask)
            glass_mask = np.where(binary == 1, 1, glass_mask)
            break

    for w_idx in range(wood_count):
        for _ in range(16):
            mask = generate_wood_mask(width, height, rng)
            binary = (mask > 0.28).astype(np.uint8)
            if binary.sum() < min_pixels:
                continue
            binary[combined_mask > 0] = 0
            if binary.sum() < min_pixels:
                continue
            effect_mask = mask * binary
            defect_np = apply_wood_effect(defect_np, effect_mask, rng)
            combined_mask = np.where(binary == 1, 2, combined_mask)
            wood_mask = np.where(binary == 1, 1, wood_mask)
            break

    defect_np = np.clip(defect_np, 0, 255).astype(np.uint8)
    context_np = defect_np.copy()
    context_np[combined_mask > 0] = 0

    masks = {
        'combined': combined_mask.astype(np.uint8),
        'binary': (combined_mask > 0).astype(np.uint8) * 255,
        'glass': glass_mask.astype(np.uint8) * 255,
        'wood': wood_mask.astype(np.uint8) * 255,
    }

    return base_np.astype(np.uint8), defect_np, context_np, masks


In [ ]:
def ensure_split_dirs(root, split):
    split_root = root / split
    for sub in ['original', 'defected', 'context', 'mask_multiclass', 'mask_inpaint', 'mask_glass', 'mask_wood']:
        (split_root / sub).mkdir(parents=True, exist_ok=True)
    return split_root

if CONVERTED_ROOT.exists():
    print(f'Removing existing converted dataset at {CONVERTED_ROOT}')
    shutil.rmtree(CONVERTED_ROOT)
CONVERTED_ROOT.mkdir(parents=True, exist_ok=True)

image_paths = [p for p in RAW_DATA_ROOT.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS]
if not image_paths:
    raise ValueError('No images found under RAW_DATA_ROOT.')

shuffled_paths = image_paths.copy()
random.Random(SEED).shuffle(shuffled_paths)
train_cut = int(0.8 * len(shuffled_paths))
val_cut = int(0.9 * len(shuffled_paths))

split_lookup = {}
for idx, path in enumerate(shuffled_paths):
    key = str(path.resolve())
    if idx < train_cut:
        split_lookup[key] = 'train'
    elif idx < val_cut:
        split_lookup[key] = 'val'
    else:
        split_lookup[key] = 'test'

records = []
for idx, image_path in enumerate(tqdm(image_paths, desc='Converting dataset')):
    split = split_lookup[str(image_path.resolve())]
    split_root = ensure_split_dirs(CONVERTED_ROOT, split)
    base_name = f'{image_path.stem}_{idx:05d}'
    rng = random.Random(SEED * 1009 + idx)

    original_np, defect_np, context_np, masks = inject_defects(image_path, rng)

    original_path = split_root / 'original' / f'{base_name}.png'
    defect_path = split_root / 'defected' / f'{base_name}.png'
    context_path = split_root / 'context' / f'{base_name}.png'
    mask_multiclass_path = split_root / 'mask_multiclass' / f'{base_name}.png'
    mask_inpaint_path = split_root / 'mask_inpaint' / f'{base_name}.png'
    mask_glass_path = split_root / 'mask_glass' / f'{base_name}.png'
    mask_wood_path = split_root / 'mask_wood' / f'{base_name}.png'

    Image.fromarray(original_np).save(original_path)
    Image.fromarray(defect_np).save(defect_path)
    Image.fromarray(context_np).save(context_path)
    Image.fromarray(masks['combined'], mode='L').save(mask_multiclass_path)
    Image.fromarray(masks['binary'], mode='L').save(mask_inpaint_path)
    Image.fromarray(masks['glass'], mode='L').save(mask_glass_path)
    Image.fromarray(masks['wood'], mode='L').save(mask_wood_path)

    mask_pixels = int((masks['binary'] > 0).sum())
    glass_pixels = int((masks['glass'] > 0).sum())
    wood_pixels = int((masks['wood'] > 0).sum())

    records.append(
        {
            'id': base_name,
            'split': split,
            'source': str(image_path.relative_to(RAW_DATA_ROOT)),
            'original': str(original_path.relative_to(CONVERTED_ROOT)),
            'defected': str(defect_path.relative_to(CONVERTED_ROOT)),
            'context': str(context_path.relative_to(CONVERTED_ROOT)),
            'mask_multiclass': str(mask_multiclass_path.relative_to(CONVERTED_ROOT)),
            'mask_binary': str(mask_inpaint_path.relative_to(CONVERTED_ROOT)),
            'mask_glass': str(mask_glass_path.relative_to(CONVERTED_ROOT)),
            'mask_wood': str(mask_wood_path.relative_to(CONVERTED_ROOT)),
            'glass_defects': 1,
            'wood_defects': 2,
            'mask_pixels': mask_pixels,
            'glass_pixels': glass_pixels,
            'wood_pixels': wood_pixels,
        }
    )

metadata_df = pd.DataFrame(records)
metadata_csv_path = CONVERTED_ROOT / 'metadata.csv'
metadata_json_path = CONVERTED_ROOT / 'metadata.json'
metadata_df.to_csv(metadata_csv_path, index=False)
metadata_df.to_json(metadata_json_path, orient='records', indent=2)

display(metadata_df.head())
print('
Dataset split counts:')
print(metadata_df.groupby('split').size())


## Quick Visual Check

Use the helper below to inspect a random sample with all derived assets.


In [ ]:
def plot_sample(row):
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    assets = [
        ('Original', row['original']),
        ('Defected', row['defected']),
        ('Context', row['context']),
        ('Inpaint Mask', row['mask_binary']),
        ('Class Mask', row['mask_multiclass']),
    ]
    for ax, (title, rel_path) in zip(axes, assets):
        img = Image.open(CONVERTED_ROOT / rel_path)
        if title.endswith('Mask'):
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

_ = plot_sample(metadata_df.sample(1, random_state=SEED).iloc[0])


## PyTorch Dataset & Dataloaders

The custom dataset reads the converted metadata and prepares tensors ready for diffusion training.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class RePaintDataset(Dataset):
    def __init__(self, metadata, split, root, image_size=256):
        self.root = Path(root)
        self.metadata = metadata[metadata['split'] == split].reset_index(drop=True)
        self.split = split
        self.image_size = image_size
        self.image_transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
            ]
        )

    def __len__(self):
        return len(self.metadata)

    def _load_image(self, relative_path):
        return Image.open(self.root / relative_path).convert('RGB')

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        target_img = self._load_image(row['original'])
        context_img = self._load_image(row['context'])

        target_tensor = self.image_transform(target_img)
        context_tensor = self.image_transform(context_img)

        mask_path = self.root / row['mask_binary']
        mask_img = Image.open(mask_path).convert('L').resize(
            (self.image_size, self.image_size), resample=Image.NEAREST
        )
        mask_tensor = torch.from_numpy(np.array(mask_img, dtype=np.float32) / 255.0).unsqueeze(0)

        multiclass_path = self.root / row['mask_multiclass']
        multiclass_img = Image.open(multiclass_path).convert('L').resize(
            (self.image_size, self.image_size), resample=Image.NEAREST
        )
        multiclass_np = np.array(multiclass_img, dtype=np.uint8)
        glass_tensor = torch.from_numpy((multiclass_np == 1).astype(np.float32)).unsqueeze(0)
        wood_tensor = torch.from_numpy((multiclass_np == 2).astype(np.float32)).unsqueeze(0)

        return {
            'target_images': target_tensor,
            'context_images': context_tensor,
            'masks': mask_tensor,
            'glass_masks': glass_tensor,
            'wood_masks': wood_tensor,
            'metadata': {
                'id': row['id'],
                'source': row['source'],
            },
        }


In [ ]:
metadata_df = pd.read_csv(CONVERTED_ROOT / 'metadata.csv')
IMAGE_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = max(1, min(8, os.cpu_count() // 2))
PIN_MEMORY = torch.cuda.is_available()

train_dataset = RePaintDataset(metadata_df, split='train', root=CONVERTED_ROOT, image_size=IMAGE_SIZE)
val_dataset = RePaintDataset(metadata_df, split='val', root=CONVERTED_ROOT, image_size=IMAGE_SIZE)
test_dataset = RePaintDataset(metadata_df, split='test', root=CONVERTED_ROOT, image_size=IMAGE_SIZE)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=max(1, NUM_WORKERS // 2), pin_memory=PIN_MEMORY)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=1, pin_memory=PIN_MEMORY)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples  : {len(val_dataset)}')
print(f'Test samples : {len(test_dataset)}')

sample_batch = next(iter(train_dataloader))
print('Batch shapes -> target:', sample_batch['target_images'].shape, 'mask:', sample_batch['masks'].shape)


## Diffusion Training Loop

The next cell initialises the UNet denoiser, scheduler, and optimiser. Tune the hyper-parameters for your hardware.


In [ ]:
import torch.nn.functional as F
from accelerate import Accelerator
from accelerate.utils import set_seed
from diffusers import UNet2DModel, DDPMScheduler

set_seed(SEED)

accelerator = Accelerator(mixed_precision='fp16')
model = UNet2DModel(
    sample_size=IMAGE_SIZE,
    in_channels=4,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(128, 256, 256),
    down_block_types=('DownBlock2D', 'DownBlock2D', 'DownBlock2D'),
    up_block_types=('UpBlock2D', 'UpBlock2D', 'UpBlock2D'),
)

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_start=0.0001,
    beta_end=0.02,
    beta_schedule='squaredcos_cap_v2',
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

NUM_EPOCHS = 20
GRADIENT_ACCUMULATION_STEPS = 1
steps_per_epoch = max(1, len(train_dataloader))
total_training_steps = NUM_EPOCHS * steps_per_epoch
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_training_steps)

model, optimizer, train_dataloader, val_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, val_dataloader
)


In [ ]:
CHECKPOINT_DIR = CONVERTED_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOG_INTERVAL = 50
global_step = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    progress_bar = tqdm(train_dataloader, disable=not accelerator.is_main_process, desc=f'Epoch {epoch:02d}')
    for step, batch in enumerate(progress_bar):
        clean_images = batch['target_images']
        context_images = batch['context_images']
        masks = batch['masks']

        noise = torch.randn_like(clean_images)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean_images.shape[0],),
            device=clean_images.device,
            dtype=torch.long,
        )
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)
        noisy_context = noisy_images * masks + context_images * (1 - masks)
        model_input = torch.cat([noisy_context, masks], dim=1)

        noise_pred = model(model_input, timesteps).sample
        loss = F.mse_loss(noise_pred, noise)

        accelerator.backward(loss)

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            lr_scheduler.step()

        global_step += 1
        if accelerator.is_main_process and global_step % LOG_INTERVAL == 0:
            progress_bar.set_postfix({'loss': loss.item()})

    accelerator.wait_for_everyone()
    model.eval()
    val_losses = []
    for batch in val_dataloader:
        with torch.no_grad():
            clean_images = batch['target_images']
            context_images = batch['context_images']
            masks = batch['masks']
            noise = torch.randn_like(clean_images)
            timesteps = torch.randint(
                0,
                noise_scheduler.config.num_train_timesteps,
                (clean_images.shape[0],),
                device=clean_images.device,
                dtype=torch.long,
            )
            noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)
            noisy_context = noisy_images * masks + context_images * (1 - masks)
            model_input = torch.cat([noisy_context, masks], dim=1)
            noise_pred = model(model_input, timesteps).sample
            val_loss = F.mse_loss(noise_pred, noise).item()
            val_losses.append(val_loss)

    if accelerator.is_main_process:
        mean_val = float(np.mean(val_losses)) if val_losses else float('nan')
        print(f'[epoch {epoch:02d}] validation loss: {mean_val:.4f}')
        checkpoint_path = CHECKPOINT_DIR / f'repaint_unet_epoch{epoch:03d}.pt'
        torch.save(accelerator.unwrap_model(model).state_dict(), checkpoint_path)


## Inference Helper

After training, run the next cells to sample the model on a validation example using the RePaint scheduler.


In [ ]:
from diffusers import RePaintScheduler

def tensor_to_pil(tensor):
    tensor = tensor.detach().cpu().clamp(-1, 1)
    tensor = (tensor + 1) / 2
    array = (tensor.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(array)

def repaint_sample(model, scheduler, context_tensor, mask_tensor, num_inference_steps=150, jump_length=10, jump_n_sample=10, generator=None):
    model = accelerator.unwrap_model(model)
    model.eval()
    generator = generator or torch.Generator(device=context_tensor.device)
    repaint_scheduler = RePaintScheduler.from_config(scheduler.config)
    repaint_scheduler.set_timesteps(num_inference_steps)
    repaint_scheduler.jump_length = jump_length
    repaint_scheduler.jump_n_sample = jump_n_sample

    with torch.no_grad():
        noisy_sample = torch.randn_like(context_tensor, generator=generator)
        known_region = context_tensor * (1 - mask_tensor)
        sample = noisy_sample * mask_tensor + known_region
        for t in repaint_scheduler.timesteps:
            model_input = torch.cat([sample, mask_tensor], dim=1)
            noise_pred = model(model_input, t).sample
            step_output = repaint_scheduler.step(noise_pred, t, sample)
            sample = step_output.prev_sample
            sample = sample * mask_tensor + known_region
    return sample


In [ ]:
inference_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)
inference_batch = next(iter(inference_loader))

context = inference_batch['context_images'].to(accelerator.device)
mask = inference_batch['masks'].to(accelerator.device)
target = inference_batch['target_images']

generated = repaint_sample(model, noise_scheduler, context, mask)

target_img = tensor_to_pil(target)
context_img = tensor_to_pil(context.cpu())
mask_img = Image.fromarray((mask.squeeze(0).detach().cpu().numpy() * 255).astype(np.uint8))
generated_img = tensor_to_pil(generated.cpu())

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, title, img in zip(
    axes,
    ['Ground Truth', 'Context', 'Mask', 'RePaint Output'],
    [target_img, context_img, mask_img, generated_img],
):
    if title == 'Mask':
        ax.imshow(img, cmap='gray')
    else:
        ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
